# Notebook 20 — Full RAG Pipeline: Retrieval + Generation
Goal: Combine retrieval with language model generation and explore prompt design.

This notebook connects:
documents → chunking → embeddings → retrieval → prompt construction → LLM answer

It also introduces **prompt design** and simple **answer evaluation**.

## Section 1 — Install and Import Libraries

In [ ]:
# If needed:
# pip install sentence-transformers scikit-learn transformers torch pandas numpy

import re
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from transformers import pipeline

## Section 2 — Example Scientific Documents

In [ ]:
documents = [
    "KRAS mutations drive abnormal MAPK pathway signaling in colorectal cancer.",
    "EGFR inhibitors such as erlotinib are commonly used in lung cancer therapy.",
    "Organoid systems allow testing of patient-specific drug responses.",
    "TP53 mutations disrupt tumor suppression and genomic stability.",
]

documents

## Section 3 — Chunking Function

In [ ]:
def chunk_text(text):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences if s]

chunks = []
for doc in documents:
    chunks.extend(chunk_text(doc))

chunks

## Section 4 — Create Embeddings

In [ ]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = embed_model.encode(chunks)

len(chunks), chunk_embeddings[0].shape

## Section 5 — Retrieval Function

In [ ]:
def retrieve(query, top_k=2):
    q_emb = embed_model.encode([query])
    sims = cosine_similarity(q_emb, chunk_embeddings)[0]
    
    idx = np.argsort(sims)[::-1][:top_k]
    
    return [chunks[i] for i in idx]

query = "What models are used to test patient drug response?"

retrieve(query)

## Section 6 — Construct a RAG Prompt

In [ ]:
def build_prompt(query, context_chunks):
    context = "\n".join(context_chunks)
    
    prompt = f"""You are a scientific assistant.
Use the provided context to answer the question.
If the answer is not contained in the context, say you do not know.

Context:
{context}

Question:
{query}

Answer:"""
    
    return prompt

context = retrieve(query)
prompt = build_prompt(query, context)

print(prompt)

## Section 7 — Load a Small Language Model

In [ ]:
generator = pipeline(
    "text-generation",
    model="distilgpt2",
    max_new_tokens=80
)

generator

## Section 8 — Generate an Answer

In [ ]:
result = generator(prompt)[0]["generated_text"]

print(result)

## Section 9 — Prompt Design Experiments
Try modifying the prompt instructions.

In [ ]:
prompt2 = f"""Answer the question using ONLY the provided context.

Context:
{context}

Question:
{query}

Answer:"""

print(generator(prompt2)[0]["generated_text"])

## Section 10 — Simple Evaluation
Check:
- Did retrieval return the right chunks?
- Did the answer use information from the context?
- Did the model hallucinate anything?

In [ ]:
print("Retrieved Context:")
for c in context:
    print("-", c)

## Section 11 — Exercises

1. Modify the prompt to require citations from the context.
2. Change the query wording and observe changes in retrieval.
3. Increase top_k retrieval to 3.
4. Try removing the 'If not in context say you do not know' instruction.
5. Evaluate whether hallucinations increase.

## Skills Gained
- building a full RAG pipeline
- connecting retrieval to generation
- designing RAG prompts
- inspecting model hallucinations
- evaluating RAG output quality